# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
import os
import chromadb
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
# TODO: Load environment variables
# load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
from chromadb.utils import embedding_functions
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL
)


In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
   name="udaplay",
   embedding_function=embedding_fn
)
@tool
def retrieve_game(query: str):
   """
   Finds most results in the vector DB
   args:
   - query: a question about game industry.
   response:
   You'll receive results as list. Each element contains:
   - Platform: like Game Boy, Playstation 5, Xbox 360...)
   - Name: Name of the Game
   - YearOfRelease: Year when that game was released for that platform
   - Description: Additional details about the game
   """
   return collection.query(query_texts=[query])


#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

from lib.parsers import PydanticOutputParser
from pydantic import BaseModel, Field
class EvaluationReport(BaseModel):
    useful: bool = Field(description="True if the retrieved documents are enough to answer the question")
    description: str = Field(description="Detailed explanation of why the documents are or are not enough")

judge_llm = LLM(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.0)

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond to the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not. "
            "Mark useful=true only if the documents directly contain the facts needed "
            "(e.g. the right game, platform and year); otherwise mark useful=false."
        )),
        UserMessage(content=f"Question: {question}\n\nRetrieved documents:\n{retrieved_docs}"),
    ]
    response = judge_llm.invoke(messages, response_format=EvaluationReport)
    return PydanticOutputParser(model_class=EvaluationReport).parse(response).model_dump()

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> dict:
    """
    Finds results in the web, used when the vector DB
    does not have enough information to answer.
    args:
    - question: a question about game industry.
    The result includes:
    - answer: short answer synthesized by Tavily (may be empty)
    - results: list of {title, url, content} for the top web pages
    """
    response = tavily_client.search(
        query=question,
        max_results=5,
        include_answer=True,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {"title": r["title"], "url": r["url"], "content": r["content"]}
            for r in response.get("results", [])
        ],
    }


### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
instructions = """
You are UdaPlay, a research assistant for the video game industry.
Answer questions about games which should contain below details:
- release dates
- platforms
- genres
- publishers
- details

Follow below setps for every question:
1. Call retrieve_game with the user's question to search internal database.
2. Call evaluate_retrieval with the original question and the retrieved documents for evaluation.
3. If the evaluation says the documents are useful, answer from them only.
4. If the evaluation says not useful, call game_web_search with the question and answer from the web results. Always include the full URL of each webpage you used.

Rules:
- Never answer from your own memory; use only tool results.
- State where the answer came from: "internal database" or "web search".
- Check the platform. A game released on one platform may not exist on another.
- If neither source has the answer, apologies and say not found rather than hallucinating.
- Keep answers short, precise and factual based on information retrived.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)


In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]
for q in questions:
    run = agent.invoke(q, session_id="udaplay")
    print(q, "\n--->", run.get_final_state()["messages"][-1].content, "\n\n\n")


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

long_term_memory = LongTermMemory(VectorStoreManager(OPENAI_API_KEY))
USER_ID = "12345"
instructions = """
You are UdaPlay, a research assistant for the video game industry.
Answer questions about games which should contain below details:
- release dates
- platforms
- genres
- publishers
- details

Follow below setps for every question:
1. Call retrieve_game with the user's question to search internal database.
2. Call evaluate_retrieval with the original question and the retrieved documents for evaluation.
3. If the evaluation says the documents are useful, answer from them only.
4. If the evaluation says not useful, call game_web_search with the question and answer from the web results. Always include the full URL of each webpage you used.

Rules:
- Never answer from your own memory; use only tool results.
- Call recall_memory with the question always. If a saved fact fully answers, use it and answer it.
- State where the answer came from: "internal database" or "web search" or "long-term memory".
- Check the platform. A game released on one platform may not exist on another.
- If neither source has the answer, apologies and say not found rather than hallucinating.
- Keep answers short, precise and factual based on information retrived.
- After you have your answer but BEFORE writing the final reply, call save_memory with one short fact that includes the game, platform and year. Only after save_memory returns, write the final answer.
"""

@tool
def save_memory(fact: str) -> str:
    """
    Stores a fact for future conversations, such as a verified answer found.
    args:
    - fact: one short self-contained sentence to remember
    """
    long_term_memory.register(MemoryFragment(content=fact, owner=USER_ID))
    return "saved"

@tool
def recall_memory(query: str) -> list[str]:
    """
    Searches previously saved facts relevant to the query.
    args:
    - query: what to look up
    """
    result = long_term_memory.search(query, owner=USER_ID, limit=3)
    return [f.content for f in result.fragments]

memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions= instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search, recall_memory, save_memory],
    temperature=0.0,
)


In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
# Here when you will re run without changing user id it will not execute the tool but use memory.
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]
for q in questions:
    run = memory_agent.invoke(q, session_id="udaplay")
    print(q, "\n--->", run.get_final_state()["messages"][-1].content, "\n\n\n")
